# Load epub book

In [3]:
# Import libraries
import os
from langchain_community.document_loaders import UnstructuredEPubLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

import chromadb
from uuid import uuid4
from chromadb.utils import embedding_functions

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [4]:
# TODO: Load document 
epub_loader = UnstructuredEPubLoader(
    file_path="day03/docs/charles-dickens_a-christmas-carol.epub"
)
doc = epub_loader.load()

[WARNING] Could not load translations for en-US
  data file translations/en.yaml not found
[WARNING] The term Abstract has no translation defined.



In [5]:
# TODO Split document
chunk_size = 1024
chunk_overlap = 50

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=chunk_size,
    chunk_overlap=chunk_overlap
)
chunks = text_splitter.split_documents(doc)

# load document, then pass to text splitter to split
# previously, we also saw load_and_split, where we pass the splitter in as arg

In [6]:
# TODO Examine chunk
print(len(chunks))
print(chunks[5])

203
page_content='External heat and cold had little influence on Scrooge. No warmth could warm, no wintry weather chill him. No wind that blew was bitterer than he, no falling snow was more intent upon its purpose, no pelting rain less open to entreaty. Foul weather didn’t know where to have him. The heaviest rain, and snow, and hail, and sleet could boast of the advantage over him in only one respect. They often “came down” handsomely, and Scrooge never did.

Nobody ever stopped him in the street to say, with gladsome looks, “My dear Scrooge, how are you? When will you come to see me?” No beggars implored him to bestow a trifle, no children asked him what it was o’clock, no man or woman ever once in all his life inquired the way to such and such a place, of Scrooge. Even the blind men’s dogs appeared to know him; and, when they saw him coming on, would tug their owners into doorways and up courts; and then would wag their tails as though they said, “No eye at all is better than an evi

# Create embeddings

In [7]:
# Why vector databases?
# Traditional DBs look for exact matches, do not recognise semantic similarity
# Vector DBs give you an approximation of the query using cosine space

# Embeddings
# Think of RGBA embeddings that represent the intensity of each colour component
# Then we can treat these as 3 or 4 dimensional vectors, plot them in space etc.
# And we can compute the similarity (distance) between colours! 
# We can define the threshold for "closeness"

# Embedding functions convert text to a vector of a specified size (regardless of length of text)

In [8]:
# TODO: Create embedding model
embed_model_name = "BAAI/bge-small-en-v1.5"
#embed_model_name = "all-MiniLM-L6-v2"

embed_model_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embed_model_name)

In [9]:
# TODO: Explore embedding model
text = "dog"

emb_text = embed_model_fn([ text ])
print(len(emb_text[0]))
# 1 word is 384

384


In [10]:
# Try with a sentence instead of 1 word
text = "GPT-5 made its debut at the end of last week"

emb_text = embed_model_fn([ text ])
print(len(emb_text[0]))

384


In [11]:
idx = 50
print(chunks[idx].page_content)

all this was visible; and which was doubtless the occasion of its using, in its duller moments, a great extinguisher for a cap, which it now held under its arm.


In [12]:
# TODO: Prepare the chunks for inserting into Chroma
# Ignore metadata, just take the text content into an array
texts = [ d.page_content for d in chunks ]
print(texts[idx])
print(len(texts))


all this was visible; and which was doubtless the occasion of its using, in its duller moments, a great extinguisher for a cap, which it now held under its arm.
203


In [13]:
# TODO: Generate PK for texts
# Generate UUIDs to use as primary keys for the chunks in the DB

texts_ids = [ str(uuid4())[:8] for _ in range(len(texts)) ]
print(texts_ids[idx])
print(len(texts_ids))

f3573d4d
203


In [14]:
# TODO: Create ephemeral Chroma client and save chunks
# "Tables" in Chroma are called collections, like in document DBs

collection_name = "epub"
# Create a Chroma client
chroma_client = chromadb.Client()
# Load embedding function (embed_model_fn)
embed_model_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name=embed_model_name)


In [15]:
# TODO: Insert the chunks into the collection
# Note: can only use one embedding function per collection

# Delete collection if it already exists
try:
    chroma_client.delete_collection(collection_name)
except:
    pass

# Create collection, specifying embed fn for the collection
epub_col = chroma_client.create_collection(
    name=collection_name,
    embedding_function=embed_model_fn
)

# If the document count <= 0, then we load
if epub_col.count() <= 0:
    print("Adding documents...")
    epub_col.add(
        documents=texts,
        ids=texts_ids
    )


Adding documents...


In [16]:
# TODO: Print number of documents in collection 
print(epub_col.count())

203


In [17]:
# TODO: Query collection 
query = "Who is Marley?"

# Can also convert images into embeddings and use as query!
results = epub_col.query(
    query_texts=[ query ], # pass text in as an array
    n_results=5 # return top 5 results
)

for k, v in results.items():
    print(f"{k}: {v}\n")
    
# Uses distances, not angles

ids: [['b3b27d71', '779db649', '9ee3cf9e', '05d72ff7', '9c1484be']]

embeddings: None

documents: [['“How now!” said Scrooge, caustic and cold as ever. “What do you want with me?”\n\n“Much!”\ufeff—Marley’s voice; no doubt about it.\n\n“Who are you?”\n\n“Ask me who I was.”\n\n“Who were you, then?” said Scrooge, raising his voice. “You’re particular, for a shade.” He was going to say “to a shade,” but substituted this, as more appropriate.\n\n“In life I was your partner, Jacob Marley.”\n\n“Can you\ufeff—can you sit down?” asked Scrooge, looking doubtfully at him.\n\n“I can.”\n\n“Do it, then.”\n\nScrooge asked the question, because he didn’t know whether a ghost so transparent might find himself in a condition to take a chair; and felt that in the event of its being impossible, it might involve the necessity of an embarrassing explanation. But the Ghost sat down on the opposite side of the fireplace, as if he were quite used to it.\n\n“You don’t believe in me,” observed the Ghost.\n\n“I d

In [18]:
for id in results['ids'][0]:
    chunk = epub_col.get(id)
    print(chunk['documents'])

['“How now!” said Scrooge, caustic and cold as ever. “What do you want with me?”\n\n“Much!”\ufeff—Marley’s voice; no doubt about it.\n\n“Who are you?”\n\n“Ask me who I was.”\n\n“Who were you, then?” said Scrooge, raising his voice. “You’re particular, for a shade.” He was going to say “to a shade,” but substituted this, as more appropriate.\n\n“In life I was your partner, Jacob Marley.”\n\n“Can you\ufeff—can you sit down?” asked Scrooge, looking doubtfully at him.\n\n“I can.”\n\n“Do it, then.”\n\nScrooge asked the question, because he didn’t know whether a ghost so transparent might find himself in a condition to take a chair; and felt that in the event of its being impossible, it might involve the necessity of an embarrassing explanation. But the Ghost sat down on the opposite side of the fireplace, as if he were quite used to it.\n\n“You don’t believe in me,” observed the Ghost.\n\n“I don’t,” said Scrooge.\n\n“What evidence would you have of my reality beyond that of your own senses?

In [19]:
# If we ask a query like "What is this story about?", 
# The vector DB would not be able to answer
# Instead, we would retrieve the relevant chunks and pass to LLM to summarise

# Question and Answer
Implement a question and answer LLM with the vector database. You will use `google/flan-t5-base` for question and answer.

You will use the following prompt template:

    ```
    Answer based on context:\n
    \n
    one or more context here
    \n
    question here
    ```

You will query Chroma to get the top 5 results, and use the results as the context. For example, if your question is `"Who is Scrooge?"`:

    ```
    Answer based on context:\n
    \n
    top 5 results from Chroma based on the question
    \n
    Who is Scrooge?
    ```
    
Do not worry about the accuracy of the result. Focus on implementing the solution. We will discuss the nuances of the solution at the end of the workshop

In [41]:
# TODO Load libraries
from transformers import AutoModelForSeq2SeqLM, GenerationConfig
import torch

model_name = "google/flan-t5-base" 

In [20]:
# TODO Create the model and tokenizer
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [ ]:
# TODO Set the question
question = "What happened to Marley?"

In [29]:
# TODO Get the top k answers
# The sequence is too long (1260), compared to the context window (512) if we use 5 contexts
# Change to use 3 contexts

top_k = 3
results = epub_col.query(
    query_texts=[ question ],
    n_results=top_k
)
print(results)

{'ids': [['2f45aa76', '9c1484be', '779db649']], 'embeddings': None, 'documents': [['“Scrooge and Marley’s, I believe,” said one of the gentlemen, referring to his list. “Have I the pleasure of addressing Mr. Scrooge, or Mr. Marley?”\n\n“Mr. Marley has been dead these seven years,” Scrooge replied. “He died seven years ago, this very night.”\n\n“We have no doubt his liberality is well represented by his surviving partner,” said the gentleman, presenting his credentials.\n\nIt certainly was; for they had been two kindred spirits. At the ominous word “liberality” Scrooge frowned, and shook his head, and handed the credentials back.\n\n“At this festive season of the year, Mr. Scrooge,” said the gentleman, taking up a pen, “it is more than usually desirable that we should make some slight provision for the poor and destitute, who suffer greatly at the present time. Many thousands are in want of common necessaries; hundreds of thousands are in want of common comforts, sir.”\n\n“Are there no 

In [30]:
# TODO Combine the contexts into a single string
context = ""
for id in results['ids'][0]:
    chunk = epub_col.get(id)['documents'][0]
    context = context + chunk
print(context)

“Scrooge and Marley’s, I believe,” said one of the gentlemen, referring to his list. “Have I the pleasure of addressing Mr. Scrooge, or Mr. Marley?”

“Mr. Marley has been dead these seven years,” Scrooge replied. “He died seven years ago, this very night.”

“We have no doubt his liberality is well represented by his surviving partner,” said the gentleman, presenting his credentials.

It certainly was; for they had been two kindred spirits. At the ominous word “liberality” Scrooge frowned, and shook his head, and handed the credentials back.

“At this festive season of the year, Mr. Scrooge,” said the gentleman, taking up a pen, “it is more than usually desirable that we should make some slight provision for the poor and destitute, who suffer greatly at the present time. Many thousands are in want of common necessaries; hundreds of thousands are in want of common comforts, sir.”

“Are there no prisons?” asked Scrooge.

“Plenty of prisons,” said the gentleman, laying down the pen again.M

In [31]:
# TODO Pass context to prompt template
prompt = f"Answer based on context:\n\n{context}\n\n{question}"

print(prompt)

Answer based on context:

“Scrooge and Marley’s, I believe,” said one of the gentlemen, referring to his list. “Have I the pleasure of addressing Mr. Scrooge, or Mr. Marley?”

“Mr. Marley has been dead these seven years,” Scrooge replied. “He died seven years ago, this very night.”

“We have no doubt his liberality is well represented by his surviving partner,” said the gentleman, presenting his credentials.

It certainly was; for they had been two kindred spirits. At the ominous word “liberality” Scrooge frowned, and shook his head, and handed the credentials back.

“At this festive season of the year, Mr. Scrooge,” said the gentleman, taking up a pen, “it is more than usually desirable that we should make some slight provision for the poor and destitute, who suffer greatly at the present time. Many thousands are in want of common necessaries; hundreds of thousands are in want of common comforts, sir.”

“Are there no prisons?” asked Scrooge.

“Plenty of prisons,” said the gentleman, l

In [35]:
# TODO Encode the prompt
# Recall that we need the output in tensor format
enc_prompt = tokenizer(
    prompt,
    return_tensors="pt"
).input_ids
print(enc_prompt)


tensor([[11801,     3,   390,    30,  2625,    10,   105,   134,  2771,    32,
           397,    11,  1571,  1306,    22,     7,     6,    27,   857,   642,
           243,    80,    13,     8,  7569,   904,     6,     3, 13215,    12,
           112,   570,     5,   105,   566,     9,   162,    27,     8,  5565,
            13,     3, 14198,  1363,     5,   180,  2771,    32,   397,     6,
            42,  1363,     5,  1571,  1306,  4697,   105,   329,    52,     5,
          1571,  1306,    65,   118,  3654,   175,  2391,   203,   642,   180,
          2771,    32,   397, 18606,     5,   105,  3845,  3977,  2391,   203,
           977,     6,    48,   182,   706,  1239,   105,  1326,    43,   150,
          3228,   112, 10215,   485,    19,   168,  7283,    57,   112,     3,
         22279,  2397,   642,   243,     8, 26266,     6,     3, 12072,   112,
         17500,     5,    94,  1852,    47,   117,    21,    79,   141,   118,
           192,   773,  1271, 16370,     5,   486,  

In [36]:
# TODO Get the answer from the model and decode it
enc_answer = model.generate(enc_prompt)
# Note answer is an array of array
answer = tokenizer.decode(enc_answer[0], skip_special_tokens=True)
print(answer)

He died


In [37]:
def generate_answer(question, top_k=3):
    results = epub_col.query(
        query_texts=[ question ],
        n_results=top_k
    )
    context = ""
    for id in results['ids'][0]:
        chunk = epub_col.get(id)['documents'][0]
        context = context + chunk
    prompt = f"Answer based on context:\n\n{context}\n\n{question}"
    enc_prompt = tokenizer(
        prompt,
        return_tensors="pt"
    ).input_ids
    enc_answer = model.generate(enc_prompt)
    # Note answer is an array of array
    answer = tokenizer.decode(enc_answer[0], skip_special_tokens=True)
    return answer


In [ ]:
# Enable sampling, play with GenerationConfig
config = GenerationConfig(
    do_sample=True,
    temperature=1.0,
    top_k=3
)
def generate_answer(question, top_k=3):
    results = epub_col.query(
        query_texts=[ question ],
        n_results=top_k
    )
    context = ""
    for id in results['ids'][0]:
        chunk = epub_col.get(id)['documents'][0]
        context = context + chunk
    prompt = f"Answer based on context:\n\n{context}\n\n{question}"
    enc_prompt = tokenizer(
        prompt,
        return_tensors="pt"
    ).input_ids
    enc_answer = model.generate(enc_prompt, generation_config=config)
    # Note answer is an array of array
    answer = tokenizer.decode(enc_answer[0], skip_special_tokens=True)
    return answer

In [43]:
print(generate_answer("Who is Marley?"))

Jacob Marley


In [44]:
print(generate_answer("Why does the Scrooge hate Christmas?"))

He doesn’t make merry himself


In [45]:
print(generate_answer("What is this story about?"))

The story relates how Scrooge and the Ghost became separated.


# Discussion

1. How does your solution perform?
2. Where do you think are the issues?
3. How can you improve it?